In [1]:
# %%
import os
import sys
import json
import re # <-- Added
from pathlib import Path

# --- Make sure your utils are imported for pathing ---
file_path = os.getcwd()
project_root = os.path.dirname(file_path)
print(f"Project Root: {project_root}")
if project_root not in sys.path:
    sys.path.append(project_root)

from dotenv import load_dotenv
import chromadb
from langchain_core.embeddings import Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma 
from langchain_core.documents import Document

# --- Import your project's agents and utils ---
from crunch_llm.code.agents import BasicAgent 
from crunch_llm.code.utils import initialize_local_data


Project Root: /oscar/data/gk/jdtoscan/LLM/Crunch_LLM


/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
analyzer_system_prompt = """
You are an expert scientific code analyst. Your task is to analyze the provided code snippet and extract key information into a structured JSON format. You must be highly specific.

- **`equation`**: Identify the specific PDE being solved (e.g., `AllenCahn`, `Burgers`, `FisherKPP`).
- **`dimensionality`**: Specify the problem's dimensions (e.g., `1D+t`, `2D+t`, `3D+t`).
- **`optimizer_scheme`**: This is critical. Identify the optimization method.
    - If you see `Adam` followed by `minimize` or `SSBroyden`, label this `Hybrid Adam+SSBroyden`.
    - If you see only `Adam`, label this `Adam`.
    - If you see only `LBFGS` or `minimize`, label this `LBFGS`.
- **`problem_type`**: Identify if it is a `forward` or `inverse` problem.

Your final output must be a single, valid JSON object. Do not add any explanatory text.

---
**Example 1 (The file you are analyzing):**

*Input Code Snippet (partial):*
```python
# ... (Adam training loop) ...
for e in pbar:
    ...
    params, states[key_params] = update_model(...)
# ... (Handoff to 2nd order) ...
initial_weights, unflatten_func = ravel_pytree(params)
result = minimize(
    fun=loss_and_gradient,
    x0=initial_weights,
    method='BFGS', # SSBroyden version
)
```

*Your Required JSON Output:*
```json
{{
  "equation": "AllenCahn",
  "dimensionality": "1D+t",
  "optimizer_scheme": "Hybrid Adam+SSBroyden",
  "problem_type": "forward"
}}
```

---
**Example 2 (Simpler case):**

*Input Code Snippet (partial):*
```python
# ... (Adam training loop) ...
for e in pbar:
    ...
    params, states[key_params] = update_model(...)
# ... (NO 2nd order method) ...
print("Training complete.")
```

*Your Required JSON Output:*
```json
{{
  "equation": "Unknown",
  "dimensionality": "Unknown",
  "optimizer_scheme": "Adam",
  "problem_type": "forward"
}}
```
"""


In [3]:

# 1. Define the exact path where you want to save the database
my_database_path ="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_pinns"

# PDF_FILE_PATH = "/users/jdtoscan/data/jdtoscan/LLM/Documents/codes/SciML/Guides/PINNs_Manual_1.pdf" #
# DB_PATH = "/users/jdtoscan/data/jdtoscan/LLM/Documents/database_manuals"
# COLLECTION_NAME = "PINNs_Manual_v1"
# SOURCE_NAME = "PINNs_Manual_1.pdf" # Metadata name


# 2. Pass that path to your function
vector_store = initialize_local_data(db_path=my_database_path,collection_name='PINNs_Codes')

# 3. CRITICAL: Initialize ANALYZER with BasicAgent
# We change this from BasicStructuredAgent to BasicAgent
# and ask for a simple text string, not JSON.
analyzer_agent_prompt_string = """
You are an expert scientific code analyst. Read the provided code snippet
and identify the following, returning a single line of text:
1. Equation (e.g., AllenCahn)
2. Dimensionality (e.g., 1D+t)
3. Optimizer (e.g., Hybrid Adam+SSBroyden)
4. Problem Type (e.g., forward)

Return this as a single-line, comma-separated string.
Example: AllenCahn, 1D+t, Hybrid Adam+SSBroyden, forward
"""

analyzer_agent = BasicAgent(
                     model='gemini-2.5-flash', 
                     model_provider='google',
                     system_prompt=analyzer_system_prompt,
                     verbose=False)

# %%
# The directory containing your code files
target_path = '/users/jdtoscan/data/jdtoscan/LLM/Documents/codes/SciML/PINNs/'


Initializing with Ollama. Model: bge-m3:567m
✅ Vector store initialized successfully from path: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_pinns
   Provider: ollama, Model: bge-m3:567m, Collection: PINNs_Codes


In [10]:
# The directory containing your code files
target_path = '/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/SciML/PINNs/'

all_results = {}
print(f"\nStarting analysis of all files in: {target_path}")

# os.walk will traverse the entire directory tree
for root, dirs, files in os.walk(target_path):
    for filename in files:
        file_path = os.path.join(root, filename)
        
        # Skip non-python files
        if not filename.endswith('.py'):
            continue
            
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                code_content = f.read()
            line_count = code_content.count('\n') + 1
            print(f"Processing: {filename} ({line_count} lines)...")
            
            # --- 1. Run Analyzer Agent ONCE on the full file ---
            print("  - Running Analyzer Agent...")
            
            # --- THIS IS THE FIX ---
            # Call BasicAgent (which returns a Dict)
            agent_result = analyzer_agent.run(code_content) 
            # Get the raw string output
            summary_string = agent_result['output'].strip()
            
            # Manually create the dictionary (no parser needed)
            try:
                parts = summary_string.split(',')
                analysis_result = {
                    "equation": parts[0].strip(),
                    "dimensionality": parts[1].strip(),
                    "optimizer_scheme": parts[2].strip(),
                    "problem_type": parts[3].strip(),
                    "filename": filename
                }
            except Exception as e:
                print(f"  - Warning: Could not parse agent summary '{summary_string}'. Error: {e}")
                analysis_result = {
                    "equation": "Unknown",
                    "dimensionality": "Unknown",
                    "optimizer_scheme": "Unknown",
                    "problem_type": "Unknown"
                }
            print(f"  - Analysis: {analysis_result}")
            # --- END FIX ---
            
            
            # --- 2. Correct Splitting & Re-assembly ---
            print("  - Splitting code into cells...")
            
            # This pattern matches '# ## Cell N:' at the START of any line
            split_pattern = r'(^\s*# ## Cell \d+:.*)'
            
            # Split using MULTILINE mode
            parts = re.split(split_pattern, code_content, flags=re.MULTILINE)
            
            cleaned_cells = []
            
            # Handle the "prologue" (anything before the first Cell marker)
            if parts[0].strip():
                cleaned_cells.append(parts[0].strip())
                
            # Loop over the split parts, taking 2 at a time
            # (This pairs the delimiter `Cell N` with its content)
            for i in range(1, len(parts), 2):
                if i + 1 < len(parts):
                    # Combine the delimiter (parts[i]) and its content (parts[i+1])
                    full_cell = parts[i] + parts[i+1]
                    cleaned_cells.append(full_cell.strip())
                else:
                    # Handle a trailing delimiter with no content
                    cleaned_cells.append(parts[i].strip())

            num_cells = len(cleaned_cells)
            # --- END CORRECTED SPLITTING ---
            
            print(f"  - Found {num_cells} cells. Ingesting each as a 'part'...")

            # --- 3. Prepare common metadata & header info ---
            base_metadata = {
                'source_file': file_path,
                'filename': filename,
                'type': 'code-notebook',
                **analysis_result # Add the (full file) analysis
            }
            
            file_ext = Path(file_path).suffix.lower()
            language_map = {'.py': 'python', '.m': 'matlab', '.jl': 'julia', '.cpp': 'cpp', '.c': 'c'}
            language = language_map.get(file_ext, 'unknown')
            comment_styles = {'python': '#', 'matlab': '%', 'julia': '#', 'c': '//', 'cpp': '//'}
            comment = comment_styles.get(language, '#')
            
            analysis_str = json.dumps(analysis_result, indent=2).replace('{', '').replace('}', '').replace('"', '')

            docs_to_add = []
            
            # --- 4. Loop through each cell and create a Document ---
            for i, cell_content in enumerate(cleaned_cells):
                part_num = i + 1
                
                # --- Extract the cell's description ---
                cell_description = "File Prologue / Imports" # Default for Part 1
                
                # Find the *first* meaningful line for the description
                if cell_content.startswith('# ## Cell'):
                     # The first line is the description
                     first_line = cell_content.split('\n', 1)[0]
                     # Clean it up: "# ## Cell 1: ..." -> "Cell 1: ..."
                     cell_description = first_line.strip().lstrip('#').strip()
                else:
                    # For the prologue, find the first line of code
                    first_line = ""
                    for line in cell_content.split('\n'):
                        if line.strip() and not line.strip().startswith('#'):
                            first_line = line
                            break
                    if first_line:
                        cell_description = first_line.strip()
                
                # --- Create metadata for this specific chunk ---
                chunk_metadata = base_metadata.copy()
                chunk_metadata['part'] = f"{part_num}/{num_cells}"
                chunk_metadata['cell_description'] = cell_description
                
                # --- Create the full header for this chunk ---
                header = f"""{comment} CODE ANALYSIS (PART {part_num} of {num_cells})
{comment} DESCRIPTION: {cell_description}
{comment} 
{comment} {analysis_str.replace(chr(10), f'{chr(10)}{comment} ')}
{comment} =======================================================\n\n"""
                
                enhanced_content = header + cell_content
                
                doc = Document(page_content=enhanced_content, metadata=chunk_metadata)
                docs_to_add.append(doc)

            # --- 5. Add all documents for this file at once ---
            if docs_to_add:
                vector_store.add_documents(docs_to_add)
                
            print(f"✅ Ingested: {filename} ({num_cells} chunks)")
            all_results[file_path] = {"analysis": analysis_result, "chunks_created": num_cells}

        except Exception as e:
            # This will catch errors from the analyzer_agent
            print(f"--- UNHANDLED ERROR analyzing {filename}: {e} ---")
            all_results[file_path] = {"error": str(e)}

# --- 6. Save and Display Results ---
output_file = 'full_code_analysis.json'
print(f"\nAnalysis complete! ✅ Saving all results to {output_file}...")
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=4)
print("Done.")


Starting analysis of all files in: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/SciML/PINNs/
Processing: Allen_Cahn.py (784 lines)...
  - Running Analyzer Agent...
  - Analysis: {'equation': '```json\n{\n  "equation": "AllenCahn"', 'dimensionality': '"dimensionality": "1D+t"', 'optimizer_scheme': '"optimizer_scheme": "Hybrid Adam+SSBroyden"', 'problem_type': '"problem_type": "forward"\n}\n```', 'filename': 'Allen_Cahn.py'}
  - Splitting code into cells...
  - Found 19 cells. Ingesting each as a 'part'...
✅ Ingested: Allen_Cahn.py (19 chunks)

Analysis complete! ✅ Saving all results to full_code_analysis.json...
Done.


In [5]:

# Assuming 'vector_store' is your initialized Chroma object
print("Fetching all documents from the vector store...")

# The .get() method returns a dictionary with IDs, documents, metadatas, etc.
# We specify what information we want to retrieve.
all_entries = vector_store.get(
    include=["metadatas", "documents"] # You can also include "embeddings"
)

# Let's print the first 5 documents found
num_entries = len(all_entries['ids'])
print(f"Found {num_entries} total entries.\n")

for i in range(min(5, num_entries)):
    doc_id = all_entries['ids'][i]
    metadata = all_entries['metadatas'][i]
    content = all_entries['documents'][i]
    
    print(f"--- Entry {i+1} ---")
    print(f"ID: {doc_id}")
    print(f"Source File: {metadata.get('source_file', 'N/A')}")
    print(f"Content Preview: {content[:500]}...") # Print first 150 chars
    print("-" * 20)


Fetching all documents from the vector store...
Found 19 total entries.

--- Entry 1 ---
ID: e9655fe6-17e4-4cd3-80a6-906384c823b6
Source File: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/SciML/PINNs/Allen_Cahn.py
Content Preview: # CODE ANALYSIS (PART 1 of 19)
# DESCRIPTION: ## Cell 1: Core Imports and Project Setup
# 
# 
#   equation: ```json\n\n  \equation\: \AllenCahn\,
#   dimensionality: \dimensionality\: \1D+t\,
#   optimizer_scheme: \optimizer_scheme\: \Hybrid Adam+SSBroyden\,
#   problem_type: \problem_type\: \forward\\n\n```,
#   filename: Allen_Cahn.py
# 
# =======================================================

# ## Cell 1: Core Imports and Project Setup
import os
import sys
import time
import random
from fun...
--------------------
--- Entry 2 ---
ID: 5b254c61-f5af-4a6e-8c26-b89adf5920d7
Source File: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/SciML/PINNs/Allen_Cahn.py
Content Preview: # CODE ANALYSIS (PART 2 of 19)
# DESCRIPTION: ## Cell 

In [8]:
# import chromadb
# import os
# from pathlib import Path
# import sys

# # --- Make sure your utils are imported for pathing ---
# # (This isn't strictly necessary but good for consistency)
# file_path = os.getcwd()
# project_root = os.path.dirname(file_path)
# if project_root not in sys.path:
#     sys.path.append(project_root)

# my_collection_name = "PINNs_Codes"

# print(f"Attempting to clear elements from collection: '{my_collection_name}'")
# print(f"From database path: {my_database_path}")

# try:
#     # --- 2. INITIALIZE THE CLIENT ---
#     client = chromadb.PersistentClient(path=my_database_path)
    
#     # --- 3. GET THE COLLECTION ---
#     collection = client.get_collection(name=my_collection_name)
    
#     # --- 4. GET ALL ELEMENT IDS ---
#     # We get all items. include=[] is efficient as we only need IDs.
#     all_items = collection.get(include=[]) 
#     all_ids = all_items['ids']
    
#     if not all_ids:
#         print(f"\nInfo: Collection '{my_collection_name}' is already empty.")
#     else:
#         # --- 5. DELETE ALL ELEMENTS BY ID ---
#         print(f"\nFound {len(all_ids)} elements. Deleting all elements...")
#         collection.delete(ids=all_ids)
#         print(f"✅ Success: All {len(all_ids)} elements have been deleted from '{my_collection_name}'.")
#         print("The collection itself still exists.")

# except chromadb.errors.CollectionNotFoundError:
#     print(f"\nInfo: Collection '{my_collection_name}' does not exist. Nothing to delete.")
# except Exception as e:
#     print(f"\nAn error occurred: {e}")
#     print("Please check the path and ChromaDB server status.")



Attempting to clear elements from collection: 'PINNs_Codes'
From database path: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_pinns

Found 19 elements. Deleting all elements...
✅ Success: All 19 elements have been deleted from 'PINNs_Codes'.
The collection itself still exists.
